In [3]:
import pandas as pd
import numpy as np

In [4]:
df= pd.read_csv("../raw_data/amazon_india_2016.csv")

In [ ]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2016_00000001,2016-01-27,CUST_2016_00002571,PROD_000124,Apple iPhone SE 16GB White,Electronics,Smartphones,Apple,101945.03,0.00,...,False,NaN,NaN,Delivered,1,2016,1,0.23,True,4.6
1,TXN_2016_00000002,2016-01-07,CUST_2016_00003513,PROD_001612,Apple Pavilion 8GB RAM Silver,Electronics,Laptops,Apple,52750.75,14.72,...,False,NaN,NaN,Delivered,1,2016,1,2.69,True,3.7
2,TXN_2016_00000003,2016-01-28,CUST_2015_00001993,PROD_001751,Realme Slate 4GB RAM Black,Electronics,Tablets,Realme,18238.6,0.00,...,False,NaN,5.0,Delivered,1,2016,1,0.59,True,3.9
3,TXN_2016_00000004,25-01-2016,CUST_2016_00003593,PROD_000154,Samsung Galaxy J7 Prime 16GB Gold,Electronics,Smartphones,Samsung,33118.39,44.96,...,True,Republic Day Sale,4.5,Delivered,1,2016,1,0.17,TRUE,3.4
4,TXN_2016_00000005,2016-01-26,CUST_2016_00015048,PROD_001710,Lenovo Tab M10 8GB RAM Black,Electronics,Tablets,Lenovo,59718.16,49.77,...,True,Republic Day Sale,NaN,Delivered,1,2016,1,0.49,True,4.1


In [ ]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(4424), 55275)

In [ ]:
df["delivery_charges"].describe()

count    50851.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: delivery_charges, dtype: float64

In [ ]:
df.drop(columns=["delivery_charges"], inplace=True)

In [ ]:
df.shape

(55275, 33)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55275 entries, 0 to 55274
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          55275 non-null  object 
 1   order_date              55275 non-null  object 
 2   customer_id             55275 non-null  object 
 3   product_id              55275 non-null  object 
 4   product_name            55275 non-null  object 
 5   category                55275 non-null  object 
 6   subcategory             55275 non-null  object 
 7   brand                   55275 non-null  object 
 8   original_price_inr      55275 non-null  object 
 9   discount_percent        55275 non-null  float64
 10  discounted_price_inr    55275 non-null  float64
 11  quantity                55275 non-null  int64  
 12  subtotal_inr            55275 non-null  float64
 13  final_amount_inr        55275 non-null  float64
 14  customer_city           55275 non-null

In [ ]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'final_amount_inr', 'customer_city',
       'customer_state', 'customer_tier', 'customer_spending_tier',
       'customer_age_group', 'payment_method', 'delivery_days',
       'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name',
       'customer_rating', 'return_status', 'order_month', 'order_year',
       'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [ ]:
df["order_date"].head(20)

0     2016-01-27
1     2016-01-07
2     2016-01-28
3     25-01-2016
4     2016-01-26
5     2016-01-05
6     2016-01-02
7     2016-01-18
8     2016-01-13
9     2016-01-03
10    2016-01-01
11    2016-01-25
12    2016-01-24
13    09/01/2016
14    2016-01-08
15    2016-01-09
16    2016-01-25
17    2016-01-27
18    2016-01-04
19    2016-01-18
Name: order_date, dtype: object

In [ ]:
df["order_date"]=(
    df["order_date"]
    .str.replace("/","-",regex=False)
    .str.replace(" ","", regex=False)
)

parts= df["order_date"].str.split("-", expand=True)
year_last= parts[2].str. len()==4
df.loc[year_last,"order_date"]=(parts[2] + "-" + parts[0] + "-" + parts[1])
parts=df["order_date"].str.split("-", expand=True)
mask= parts[1].astype(int)>12
df.loc[mask, "order_date"]= (parts[0]+"-"+parts[2]+"-"+parts[1])
df["order_date"]= pd.to_datetime(df["order_date"], errors= "coerce")

In [ ]:
df["order_date"].min(), df["order_date"].max()


(Timestamp('2016-01-01 00:00:00'), Timestamp('2016-12-31 00:00:00'))

In [ ]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [ ]:
df["original_price_inr"].head(20)

0       101945.03
1        52750.75
2         18238.6
3        33118.39
4        59718.16
5        96021.61
6        46161.87
7       101282.29
8         24101.7
9       101166.56
10       85543.31
11       89153.93
12       42755.83
13       23059.33
14       39074.35
15       40462.28
16       98077.11
17       43575.44
18    ₹121,441.03
19       36633.45
Name: original_price_inr, dtype: object

In [ ]:
df["original_price_inr"]= df["original_price_inr"].astype(str)
df["original_price_inr"] = df["original_price_inr"].str.replace(" ","",regex=False)
df["original_price_inr"]= df["original_price_inr"].str.replace(",", "",regex=False)
df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)
df["original_price_inr"]= pd.to_numeric(df["original_price_inr"], errors="coerce")

In [ ]:
df["original_price_inr"].unique()[:20]

array([101945.03,  52750.75,  18238.6 ,  33118.39,  59718.16,  96021.61,
        46161.87, 101282.29,  24101.7 , 101166.56,  85543.31,  89153.93,
        42755.83,  23059.33,  39074.35,  40462.28,  98077.11,  43575.44,
       121441.03,  36633.45])

In [ ]:
df["original_price_inr"].dtypes

dtype('float64')

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [ ]:
df["customer_rating"]=df["customer_rating"].astype(str)
df["customer_rating"]= df["customer_rating"].str.replace("stars","",regex=False)
df["customer_rating"]= df["customer_rating"].str.split("/").str[0]
df["customer_rating"]= pd.to_numeric(df["customer_rating"],errors="coerce")


In [ ]:
df["customer_rating"].describe()

count    38511.000000
mean         4.316234
std          0.575353
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [ ]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    12517
5.0    10189
4.0     9560
3.5     3952
3.0     2293
Name: count, dtype: int64

In [ ]:
df["customer_rating"].isna().sum()

np.int64(16764)

In [ ]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    16764
4.5    12517
5.0    10189
4.0     9560
3.5     3952
3.0     2293
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [ ]:
df["customer_city"]= df["customer_city"].str.strip().str.lower()

In [ ]:
df["customer_city"].unique()

array(['pune', 'lucknow', 'mumbai', 'ahmedabad', 'delhi', 'chennai',
       'surat', 'vadodara', 'coimbatore', 'kolkata', 'indore',
       'saharanpur', 'chandigarh', 'bangalore', 'patna', 'madras',
       'bhubaneswar', 'jaipur', 'nagpur', 'allahabad', 'kanpur',
       'visakhapatnam', 'hyderabad', 'ludhiana', 'varanasi', 'kochi',
       'moradabad', 'meerut', 'bareilly', 'new delhi', 'aligarh',
       'gorakhpur', 'bengalore', 'chenai', 'bengaluru', 'calcutta',
       'banglore', 'delhi ncr', 'bombay', 'mumba'], dtype=object)

In [ ]:
city_map = {
    "madras": "chennai",
    "chenai": "chennai",

    "calcutta": "kolkata",

    "bombay": "mumbai",
    "mumba": "mumbai",

    "bengalore": "bangalore",
    "banglore": "bangalore",
    "bengaluru": "bangalore",

    "new delhi": "delhi",
    "delhi ncr": "delhi"
}

In [ ]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [ ]:
df["customer_city"]= df["customer_city"].str.title()

In [ ]:
df["customer_city"].value_counts().head()

customer_city
Mumbai       8870
Delhi        7715
Bangalore    6399
Chennai      5260
Kolkata      4187
Name: count, dtype: int64

In [ ]:
df["customer_city"].unique()

array(['Pune', 'Lucknow', 'Mumbai', 'Ahmedabad', 'Delhi', 'Chennai',
       'Surat', 'Vadodara', 'Coimbatore', 'Kolkata', 'Indore',
       'Saharanpur', 'Chandigarh', 'Bangalore', 'Patna', 'Bhubaneswar',
       'Jaipur', 'Nagpur', 'Allahabad', 'Kanpur', 'Visakhapatnam',
       'Hyderabad', 'Ludhiana', 'Varanasi', 'Kochi', 'Moradabad',
       'Meerut', 'Bareilly', 'Aligarh', 'Gorakhpur'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [ ]:
bool_candidates=[]
bool_values={"true", "false","y","n","yes", "no","0","1"}

for col in df.columns:
    vals= set(df[col].astype(str). str.lower().dropna().unique())
    if vals & bool_values:
        bool_candidates.append(col)
bool_candidates



['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [ ]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [5]:
df["category"].value_counts().head(20)

category
Electronics                  55195
Electronics & Accessories       26
Electronic                      24
ELECTRONICS                     15
Electronicss                    15
Name: count, dtype: int64

In [6]:
df["category"] = df["category"].str.strip().str.lower()


In [9]:
category_map = {
    "electronic": "electronics",
    "electronics": "electronics",
    "electronics & accessories": "electronics",
    "electronicss": "electronics"
}

In [10]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [11]:
df["category"].value_counts()

category
Electronics    55275
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [ ]:
df["delivery_days"].unique()

array(['3', '15', '5', '4', '7', '6', '0', '1', '1-2 days', '-1',
       'Same Day', '2', 'Express'], dtype=object)

In [ ]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [ ]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [ ]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [ ]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [ ]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [ ]:
df["delivery_days"].unique()

array([ 3., 15.,  5.,  4.,  7.,  6.,  0.,  1., nan,  2.])

In [ ]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [ ]:
df["delivery_days"].describe()

count    55275.000000
mean         4.273813
std          1.352905
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         15.000000
Name: delivery_days, dtype: float64

In [ ]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [ ]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [ ]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [ ]:
duplicates.shape

(538, 33)

In [ ]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
154,TXN_2016_00000155,2016-01-07,CUST_2016_00015802,PROD_001728,Xiaomi Slate 4GB RAM Silver,Electronics,Tablets,Xiaomi,19495.69,14.79,...,False,NaN,NaN,Returned,1,2016,1,0.49,True,4.2
227,TXN_2016_00000228,2016-01-04,CUST_2015_00004528,PROD_000053,OnePlus OnePlus 2 32GB Black,Electronics,Smartphones,OnePlus,101191.52,0.00,...,False,NaN,NaN,Delivered,1,2016,1,0.20,True,4.6
253,TXN_2016_00000254,2016-01-19,CUST_2016_00011800,PROD_000196,Motorola Moto G4 16GB Gold,Electronics,Smartphones,Motorola,23992.60,0.00,...,False,NaN,4.5,Delivered,1,2016,1,0.16,True,3.8
567,TXN_2016_00000568,2016-01-09,CUST_2015_00008529,PROD_000037,Samsung Galaxy Note 5 16GB Black,Electronics,Smartphones,Samsung,89153.93,18.79,...,False,NaN,4.0,Returned,1,2016,1,0.22,True,3.8
714,TXN_2016_00000715,2016-01-26,CUST_2015_00001854,PROD_001957,Noise Sports Watch Premium,Electronics,Smart Watch,Noise,37664.98,15.60,...,True,Republic Day Sale,5.0,Delivered,1,2016,1,0.03,False,4.2


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df = df.drop_duplicates()

In [ ]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
10417,TXN_2016_00010418,2016-04-03,CUST_2015_00000217,PROD_000065,Xiaomi Mi 4i 32GB Black,Electronics,Smartphones,Xiaomi,24737.54,52.79,...,True,Holi Festival,4.0,Delivered,3,2016,1,0.16,True,4.5
55053,TXN_2016_00010418_DUP,2016-04-03,CUST_2015_00000217,PROD_000065,Xiaomi Mi 4i 32GB Black,Electronics,Smartphones,Xiaomi,24737.54,52.79,...,True,Holi Festival,4.0,Delivered,3,2016,1,0.16,True,4.5
20516,TXN_2016_00020517,2016-06-09,CUST_2015_00000372,PROD_001922,Fitbit Watch,Electronics,Smart Watch,Fitbit,48823.32,12.46,...,False,NaN,4.5,Delivered,6,2016,2,0.03,True,3.0
55118,TXN_2016_00020517_DUP,2016-06-09,CUST_2015_00000372,PROD_001922,Fitbit Watch,Electronics,Smart Watch,Fitbit,48823.32,12.46,...,False,NaN,4.5,Delivered,6,2016,2,0.03,True,3.0
29887,TXN_2016_00029888,2016-08-22,CUST_2015_00000405,PROD_001680,Apple Tab M10 4GB RAM Silver,Electronics,Tablets,Apple,23110.53,0.00,...,False,NaN,NaN,Delivered,8,2016,3,0.66,True,4.2
55156,TXN_2016_00029888_DUP,2016-08-22,CUST_2015_00000405,PROD_001680,Apple Tab M10 4GB RAM Silver,Electronics,Tablets,Apple,23110.53,0.00,...,False,NaN,NaN,Delivered,8,2016,3,0.66,True,4.2
22628,TXN_2016_00022629,2016-06-24,CUST_2015_00000511,PROD_000015,Apple iPhone 6 Plus 16GB Blue,Electronics,Smartphones,Apple,188264.78,58.95,...,True,Back to School,NaN,Delivered,6,2016,2,0.20,False,4.1
55049,TXN_2016_00022629_DUP,2016-06-24,CUST_2015_00000511,PROD_000015,Apple iPhone 6 Plus 16GB Blue,Electronics,Smartphones,Apple,188264.78,58.95,...,True,Back to School,NaN,Delivered,6,2016,2,0.20,False,4.1
20218,TXN_2016_00020219,2016-06-29,CUST_2015_00000689,PROD_001728,Xiaomi Slate 4GB RAM Silver,Electronics,Tablets,Xiaomi,19495.69,48.03,...,True,Back to School,4.0,Delivered,6,2016,2,0.49,True,4.2
55021,TXN_2016_00020219_DUP,2016-06-29,CUST_2015_00000689,PROD_001728,Xiaomi Slate 4GB RAM Silver,Electronics,Tablets,Xiaomi,19495.69,48.03,...,True,Back to School,4.0,Delivered,6,2016,2,0.49,True,4.2


In [ ]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00000217  PROD_000065  2016-04-03  24737.54              2
CUST_2015_00000372  PROD_001922  2016-06-09  48823.32              2
CUST_2015_00000405  PROD_001680  2016-08-22  23110.53              2
CUST_2015_00000511  PROD_000015  2016-06-24  188264.78             2
CUST_2015_00000689  PROD_001728  2016-06-29  19495.69              2
CUST_2015_00000962  PROD_000144  2016-08-11  128209.35             2
CUST_2015_00001007  PROD_000169  2016-11-30  89606.37              2
CUST_2015_00001017  PROD_001884  2016-09-19  40989.63              2
CUST_2015_00001063  PROD_001899  2016-12-13  47276.62              2
CUST_2015_00001178  PROD_000042  2016-04-01  165072.38             2
dtype: int64

In [ ]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [ ]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [ ]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2016_00015802,PROD_001728,1,2
1,CUST_2015_00004528,PROD_000053,1,2
2,CUST_2016_00011800,PROD_000196,1,2
3,CUST_2015_00008529,PROD_000037,1,2
4,CUST_2015_00001854,PROD_001957,2,2


In [ ]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [ ]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [ ]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [ ]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [ ]:
# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers (100x decimal error) ────────────
subcategory_caps = {
    "Smart Watch":        50000,
    "Tablets":            85000,
    "Smartphones":        180000,
    "Laptops":            200000,
    "TV & Entertainment": 300000,
    "Audio":              50000,
}

outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")


Negative prices fixed: 132


Outliers fixed: 844
                        min        max      mean       50%
subcategory                                               
Audio               1763.32   38618.00  16110.56  17910.54
Laptops             2955.59  197037.46  85798.24  70071.42
Smart Watch         2021.38   48823.32  36143.85  40989.63
Smartphones         1815.27  174671.00  64154.92  47912.38
TV & Entertainment  3867.83  234409.14  80550.44  49921.33
Tablets             2154.10   83522.73  45958.32  48696.83

NaN in final_amount_inr:   1635
Negative prices remaining: 0


In [ ]:
print(f"NaN in subtotal_inr:         {df['subtotal_inr'].isna().sum()}")
print(f"NaN in discounted_price_inr: {df['discounted_price_inr'].isna().sum()}")
print(f"NaN in quantity:             {df['quantity'].isna().sum()}")
print(f"NaN in discount_percent:     {df['discount_percent'].isna().sum()}")

NaN in subtotal_inr:         1635
NaN in discounted_price_inr: 1635
NaN in quantity:             0
NaN in discount_percent:     0


In [ ]:
# Drop rows with NaN original_price_inr
df = df.dropna(subset=["original_price_inr"]).copy()  # ← added .copy()

# Recalculate
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]




In [ ]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")


Smart Watch: ✅ All within cap

Tablets: ✅ All within cap

Smartphones: ✅ All within cap

Laptops: ✅ All within cap

TV & Entertainment: ✅ All within cap

Audio: ✅ All within cap


Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [ ]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
COD            37591
Credit Card     7557
Debit Card      5288
Net Banking     2094
UPI             1110
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [ ]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [ ]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [ ]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
29.27481746673584 MB


In [ ]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

Total columns with NaN: 2
Total rows in dataset: 53640

NaN counts and percentages:
                 NaN Count  Percentage
festival_name        36668       68.36
customer_rating      16298       30.38


In [ ]:
df.to_csv("data_cleaning_2016.csv", index=False)
print("File saved successfully!")

File saved successfully!
